# Phase 3 — Model Training and Evaluation

This notebook trains and evaluates the vehicle occupancy forecasting models.

The first model is a single-horizon XGBoost regressor for predicting active-vehicle occupancy three hours ahead.

The engineered dataset is loaded from `supervised_3h.csv`.

The timestamp is retained for chronological ordering and evaluation but is excluded from the model feature matrix.

In [1]:
import pandas as pd
import numpy as np

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [2]:
df = pd.read_csv(
    "supervised_3h.csv",
    parse_dates=["timestamp"]
)

print("Shape:", df.shape)
print("Date range:", df["timestamp"].min(), "→", df["timestamp"].max())

Shape: (4194, 30)
Date range: 2026-08-02 00:00:00 → 2026-09-18 07:15:00


In [3]:
display(df.head())
print(df.dtypes)

,timestamp,active_vehicles,lag_15m,lag_30m,lag_1h,lag_2h,lag_3h,lag_4h,lag_6h,lag_24h,...,occupancy_change_1h,occupancy_change_2h,rolling_mean_occupancy_1h,rolling_mean_arrivals_1h,rolling_mean_exits_1h,rolling_mean_net_flow_1h,hour,day_of_week,is_weekend,target
0,2026-08-02 00:00:00,86.0,97.0,95.0,96.0,102.0,113.0,142.0,151.0,17.0,...,-10.0,-16.0,97.25,13.25,14.50,-1.25,0,6,1,125.0
1,2026-08-02 00:15:00,73.0,86.0,97.0,101.0,106.0,101.0,133.0,164.0,24.0,...,-28.0,-33.0,94.75,13.50,16.00,-2.50,0,6,1,143.0
2,2026-08-02 00:30:00,75.0,73.0,86.0,95.0,110.0,90.0,111.0,158.0,34.0,...,-20.0,-35.0,87.75,9.75,16.75,-7.00,0,6,1,157.0
3,2026-08-02 00:45:00,84.0,75.0,73.0,97.0,102.0,92.0,103.0,171.0,51.0,...,-13.0,-18.0,82.75,9.75,14.75,-5.00,0,6,1,161.0
4,2026-08-02 01:00:00,83.0,84.0,75.0,86.0,96.0,102.0,113.0,159.0,78.0,...,-3.0,-13.0,79.50,10.25,13.50,-3.25,1,6,1,147.0


timestamp                    datetime64[us]
active_vehicles                     float64
lag_15m                             float64
lag_30m                             float64
lag_1h                              float64
lag_2h                              float64
lag_3h                              float64
lag_4h                              float64
lag_6h                              float64
lag_24h                             float64
arrivals_15m                        float64
arrivals_30m                        float64
arrivals_1h                         float64
exits_15m                           float64
exits_30m                           float64
exits_1h                            float64
net_flow_15m                        float64
net_flow_30m                        float64
net_flow_1h                         float64
occupancy_change_15m                float64
occupancy_change_1h                 float64
occupancy_change_2h                 float64
rolling_mean_occupancy_1h       

## 1. Prepare Features and Target

The timestamp is retained separately for chronological data splitting and evaluation but is excluded from the model inputs.

The 28 engineered numerical features are used directly as XGBoost inputs without normalization or standardization because the baseline model is tree-based.

The target is the active-vehicle occupancy three hours into the future.

In [4]:
feature_columns = [
    column for column in df.columns
    if column not in ["timestamp", "target"]
]

X = df[feature_columns]
y = df["target"]

print("Number of features:", len(feature_columns))
print("X shape:", X.shape)
print("y shape:", y.shape)

Number of features: 28
X shape: (4194, 28)
y shape: (4194,)


In [5]:
print("Missing values in X:", X.isna().sum().sum())
print("Missing values in y:", y.isna().sum())

Missing values in X: 0
Missing values in y: 0


In [6]:
train_boundary = pd.Timestamp("2026-09-05 04:45:00")
validation_boundary = pd.Timestamp("2026-09-11 18:00:00")

train_mask = df["timestamp"] < train_boundary

validation_mask = (
    (df["timestamp"] >= train_boundary) &
    (df["timestamp"] < validation_boundary)
)

test_mask = df["timestamp"] >= validation_boundary

X_train = X.loc[train_mask]
y_train = y.loc[train_mask]

X_val = X.loc[validation_mask]
y_val = y.loc[validation_mask]

X_test = X.loc[test_mask]
y_test = y.loc[test_mask]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (2935, 28)
X_val: (629, 28)
X_test: (630, 28)


## 2. Train the 3-Hour XGBoost Baseline

The baseline model is an XGBoost regressor trained to predict active-vehicle occupancy three hours ahead.

The model is trained only on the chronological training period. The validation period is used for early stopping and model selection.

No feature scaling or normalization is applied because the model is tree-based.

In [7]:
model_3h = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
    early_stopping_rounds=50
)

model_3h.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print("Best iteration:", model_3h.best_iteration)

Best iteration: 24


In [8]:
y_val_pred = model_3h.predict(X_val)

val_mae = mean_absolute_error(y_val, y_val_pred)
val_rmse = mean_squared_error(y_val, y_val_pred) ** 0.5

print(f"Validation MAE: {val_mae:.2f} vehicles")
print(f"Validation RMSE: {val_rmse:.2f} vehicles")

Validation MAE: 33.71 vehicles
Validation RMSE: 42.92 vehicles


In [9]:
y_test_pred = model_3h.predict(X_test)

test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = mean_squared_error(y_test, y_test_pred) ** 0.5

print(f"Test MAE: {test_mae:.2f} vehicles")
print(f"Test RMSE: {test_rmse:.2f} vehicles")

Test MAE: 39.92 vehicles
Test RMSE: 50.82 vehicles


## 1. Load Multi-Horizon Dataset

The multi-horizon dataset contains the same engineered features used by the 3-hour baseline, plus `horizon_minutes`.

The `horizon_minutes` feature tells the model which future prediction horizon is requested.

The timestamp is retained for chronological splitting but is excluded from the model features.

In [10]:
df_multi = pd.read_csv(
    "supervised_multihorizon.csv",
    parse_dates=["timestamp"]
)

print("Shape:", df_multi.shape)
print("Date range:", df_multi["timestamp"].min(), "→", df_multi["timestamp"].max())
print(
    "\nSamples by horizon:\n",
    df_multi["horizon_minutes"].value_counts().sort_index()
)

Shape: (25206, 31)
Date range: 2026-08-02 00:00:00 → 2026-09-18 10:00:00

Samples by horizon:
 horizon_minutes
15     4216
30     4214
60     4210
120    4202
180    4194
360    4170
Name: count, dtype: int64


In [11]:
df_multi = pd.read_csv(
    "supervised_multihorizon.csv",
    parse_dates=["timestamp"]
)

print("Shape:", df_multi.shape)
print("Date range:", df_multi["timestamp"].min(), "→", df_multi["timestamp"].max())
print(
    "\nSamples by horizon:\n",
    df_multi["horizon_minutes"].value_counts().sort_index()
)

Shape: (25206, 31)
Date range: 2026-08-02 00:00:00 → 2026-09-18 10:00:00

Samples by horizon:
 horizon_minutes
15     4216
30     4214
60     4210
120    4202
180    4194
360    4170
Name: count, dtype: int64


In [12]:
base_feature_columns = [
    column for column in df_multi.columns
    if column not in ["timestamp", "target"]
]

X_multi = df_multi[base_feature_columns]
y_multi = df_multi["target"]

print("Number of features:", len(base_feature_columns))
print("X shape:", X_multi.shape)
print("y shape:", y_multi.shape)

Number of features: 29
X shape: (25206, 29)
y shape: (25206,)


In [13]:
train_boundary = pd.Timestamp("2026-09-05 04:45:00")
validation_boundary = pd.Timestamp("2026-09-11 18:00:00")

train_mask = df_multi["timestamp"] < train_boundary

validation_mask = (
    (df_multi["timestamp"] >= train_boundary) &
    (df_multi["timestamp"] < validation_boundary)
)

test_mask = df_multi["timestamp"] >= validation_boundary

X_multi_train = X_multi.loc[train_mask]
y_multi_train = y_multi.loc[train_mask]

X_multi_val = X_multi.loc[validation_mask]
y_multi_val = y_multi.loc[validation_mask]

X_multi_test = X_multi.loc[test_mask]
y_multi_test = y_multi.loc[test_mask]

print("Train:", X_multi_train.shape)
print("Validation:", X_multi_val.shape)
print("Test:", X_multi_test.shape)

Train: (17631, 29)
Validation: (3774, 29)
Test: (3801, 29)


## 4. Train the Multi-Horizon XGBoost Model

A single XGBoost regressor is trained across all selected forecasting horizons.

The `horizon_minutes` feature tells the model which future horizon is being requested.

The same model therefore learns:

features + horizon → future occupancy

In [14]:
model_multi = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
    early_stopping_rounds=50
)

model_multi.fit(
    X_multi_train,
    y_multi_train,
    eval_set=[(X_multi_val, y_multi_val)],
    verbose=False
)

print("Best iteration:", model_multi.best_iteration)
print(
    "Boosted rounds:",
    model_multi.get_booster().num_boosted_rounds()
)

Best iteration: 36
Boosted rounds: 87


In [15]:
y_multi_val_pred = model_multi.predict(X_multi_val)
y_multi_test_pred = model_multi.predict(X_multi_test)

## 5. Multi-Horizon Evaluation

The single multi-horizon model is evaluated separately for each forecasting horizon.

MAE and RMSE are calculated independently for the validation and test periods so that performance can be compared across horizons and against the single-horizon baseline.

In [16]:
validation_results = []
test_results = []

for horizon in sorted(df_multi["horizon_minutes"].unique()):
    val_mask = (
        df_multi.loc[validation_mask, "horizon_minutes"] == horizon
    )

    test_mask_h = (
        df_multi.loc[test_mask, "horizon_minutes"] == horizon
    )

    y_val_h = y_multi_val.loc[val_mask]
    pred_val_h = y_multi_val_pred[val_mask]

    y_test_h = y_multi_test.loc[test_mask_h]
    pred_test_h = y_multi_test_pred[test_mask_h]

    validation_results.append({
        "horizon_minutes": horizon,
        "MAE": mean_absolute_error(y_val_h, pred_val_h),
        "RMSE": mean_squared_error(y_val_h, pred_val_h) ** 0.5
    })

    test_results.append({
        "horizon_minutes": horizon,
        "MAE": mean_absolute_error(y_test_h, pred_test_h),
        "RMSE": mean_squared_error(y_test_h, pred_test_h) ** 0.5
    })

multi_val_results = pd.DataFrame(validation_results)
multi_test_results = pd.DataFrame(test_results)

print("Validation:")
display(multi_val_results)

print("Test:")
display(multi_test_results)

Validation:


,horizon_minutes,MAE,RMSE
0,15,12.116355,14.866720
1,30,14.649283,18.288603
2,60,20.489120,25.659754
3,120,27.978393,35.930565
4,180,33.525740,43.276845
5,360,45.142228,57.643531


Test:


,horizon_minutes,MAE,RMSE
0,15,14.821697,18.388345
1,30,18.476474,22.584855
2,60,25.043254,30.599066
3,120,33.634450,42.810942
4,180,39.971679,51.908332
5,360,45.404426,62.658984


In [17]:
model_multi.save_model("xgb_multihorizon.json")

In [18]:
df_6h = df_multi[df_multi["horizon_minutes"] == 360].copy()

print("6-hour dataset:", df_6h.shape)
print(
    "Date range:",
    df_6h["timestamp"].min(),
    "→",
    df_6h["timestamp"].max()
)

6-hour dataset: (4170, 31)
Date range: 2026-08-02 00:00:00 → 2026-09-18 04:15:00


In [19]:
feature_columns_6h = [
    column
    for column in df_6h.columns
    if column not in ["timestamp", "target", "horizon_minutes"]
]

X_6h = df_6h[feature_columns_6h]
y_6h = df_6h["target"]

print("Number of features:", len(feature_columns_6h))
print("X shape:", X_6h.shape)
print("y shape:", y_6h.shape)
print("Missing values in X:", X_6h.isna().sum().sum())
print("Missing values in y:", y_6h.isna().sum())

Number of features: 28
X shape: (4170, 28)
y shape: (4170,)
Missing values in X: 0
Missing values in y: 0


In [20]:
train_mask_6h = df_6h["timestamp"] < train_boundary

validation_mask_6h = (
    (df_6h["timestamp"] >= train_boundary) &
    (df_6h["timestamp"] < validation_boundary)
)

test_mask_6h = df_6h["timestamp"] >= validation_boundary


X_6h_train = X_6h.loc[train_mask_6h]
y_6h_train = y_6h.loc[train_mask_6h]

X_6h_val = X_6h.loc[validation_mask_6h]
y_6h_val = y_6h.loc[validation_mask_6h]

X_6h_test = X_6h.loc[test_mask_6h]
y_6h_test = y_6h.loc[test_mask_6h]


print("Train:", X_6h_train.shape)
print("Validation:", X_6h_val.shape)
print("Test:", X_6h_test.shape)

Train: (2923, 28)
Validation: (629, 28)
Test: (618, 28)


In [21]:
from xgboost import XGBRegressor

model_6h = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
    early_stopping_rounds=50
)

model_6h.fit(
    X_6h_train,
    y_6h_train,
    eval_set=[(X_6h_val, y_6h_val)],
    verbose=False
)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",50
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [22]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Validation predictions
y_6h_val_pred = model_6h.predict(X_6h_val)

val_6h_mae = mean_absolute_error(y_6h_val, y_6h_val_pred)
val_6h_rmse = np.sqrt(mean_squared_error(y_6h_val, y_6h_val_pred))

# Test predictions
y_6h_test_pred = model_6h.predict(X_6h_test)

test_6h_mae = mean_absolute_error(y_6h_test, y_6h_test_pred)
test_6h_rmse = np.sqrt(mean_squared_error(y_6h_test, y_6h_test_pred))

print("6-hour Validation MAE:", val_6h_mae)
print("6-hour Validation RMSE:", val_6h_rmse)

print("6-hour Test MAE:", test_6h_mae)
print("6-hour Test RMSE:", test_6h_rmse)

6-hour Validation MAE: 40.78382856553992
6-hour Validation RMSE: 50.622897729087164
6-hour Test MAE: 47.0702009725725
6-hour Test RMSE: 61.63261901439453


In [23]:
print("Current multi-horizon model parameters:")
print(model_multi.get_params())

Current multi-horizon model parameters:
{'objective': 'reg:squarederror', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': 0.8, 'device': None, 'early_stopping_rounds': 50, 'enable_categorical': True, 'eval_metric': None, 'feature_types': None, 'feature_weights': None, 'gamma': None, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': 0.05, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': 6, 'max_leaves': None, 'min_child_weight': None, 'missing': nan, 'monotone_constraints': None, 'multi_strategy': None, 'n_estimators': 1000, 'n_jobs': -1, 'num_parallel_tree': None, 'random_state': 42, 'reg_alpha': None, 'reg_lambda': 1.0, 'sampling_method': None, 'scale_pos_weight': None, 'subsample': 0.8, 'tree_method': 'hist', 'validate_parameters': None, 'verbosity': None}


In [24]:
print("Baseline multi-horizon test results:")
print(multi_test_results)

Baseline multi-horizon test results:
   horizon_minutes        MAE       RMSE
0               15  14.821697  18.388345
1               30  18.476474  22.584855
2               60  25.043254  30.599066
3              120  33.634450  42.810942
4              180  39.971679  51.908332
5              360  45.404426  62.658984


In [25]:
from scipy.stats import randint, uniform

param_distributions = {
    "max_depth": randint(3, 9),
    "min_child_weight": randint(1, 11),
    "learning_rate": uniform(0.02, 0.08),
    "subsample": uniform(0.7, 0.3),
    "colsample_bytree": uniform(0.7, 0.3),
    "gamma": uniform(0.0, 0.2),
    "reg_alpha": uniform(0.0, 0.5),
    "reg_lambda": uniform(0.5, 4.5)
}

print("Hyperparameter search space created.")

Hyperparameter search space created.


In [26]:
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
import numpy as np
import pandas as pd

n_iter = 30

random_configs = list(
    ParameterSampler(
        param_distributions,
        n_iter=n_iter,
        random_state=42
    )
)

search_results = []

for i, params in enumerate(random_configs, start=1):

    model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=1500,
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
        early_stopping_rounds=50,
        **params
    )

    model.fit(
        X_multi_train,
        y_multi_train,
        eval_set=[(X_multi_val, y_multi_val)],
        verbose=False
    )

    val_pred = model.predict(X_multi_val)
    val_mae = mean_absolute_error(y_multi_val, val_pred)

    search_results.append({
        "iteration": i,
        "validation_mae": val_mae,
        "best_iteration": model.best_iteration,
        **params
    })

    print(
        f"Iteration {i:02d}/{n_iter} | "
        f"Validation MAE: {val_mae:.4f} | "
        f"Best iteration: {model.best_iteration}"
    )

search_results = pd.DataFrame(search_results)
search_results = search_results.sort_values(
    "validation_mae"
).reset_index(drop=True)

print("\nTop configurations:")
print(search_results.head(10))

Iteration 01/30 | Validation MAE: 26.2601 | Best iteration: 20
Iteration 02/30 | Validation MAE: 25.5988 | Best iteration: 19
Iteration 03/30 | Validation MAE: 25.8387 | Best iteration: 38
Iteration 04/30 | Validation MAE: 25.6945 | Best iteration: 30
Iteration 05/30 | Validation MAE: 26.3196 | Best iteration: 31
Iteration 06/30 | Validation MAE: 25.9148 | Best iteration: 47
Iteration 07/30 | Validation MAE: 26.2074 | Best iteration: 19
Iteration 08/30 | Validation MAE: 25.1177 | Best iteration: 40
Iteration 09/30 | Validation MAE: 25.9442 | Best iteration: 31
Iteration 10/30 | Validation MAE: 26.0306 | Best iteration: 59
Iteration 11/30 | Validation MAE: 26.0150 | Best iteration: 78
Iteration 12/30 | Validation MAE: 26.4150 | Best iteration: 20
Iteration 13/30 | Validation MAE: 25.8497 | Best iteration: 20
Iteration 14/30 | Validation MAE: 25.9493 | Best iteration: 32
Iteration 15/30 | Validation MAE: 25.6330 | Best iteration: 29
Iteration 16/30 | Validation MAE: 25.5242 | Best iterat

In [27]:
print(search_results.head(10).to_string(index=False))

 iteration  validation_mae  best_iteration  colsample_bytree    gamma  learning_rate  max_depth  min_child_weight  reg_alpha  reg_lambda  subsample
         8       25.117701              40          0.952685 0.089951       0.051612          6                 8   0.285222    2.843754   0.988352
        16       25.524213              37          0.964140 0.124871       0.043651          7                 6   0.228267    1.482982   0.824953
        28       25.531879              61          0.933963 0.128406       0.026731          6                 6   0.334494    3.113090   0.811685
        24       25.584781              22          0.766072 0.142230       0.084760          6                 7   0.087183    3.609220   0.816021
         2       25.598843              19          0.959853 0.120223       0.076646          8                 5   0.484955    4.245992   0.763702
        26       25.613942              26          0.897995 0.163444       0.064416          6                 

In [28]:
best_params = {
    "max_depth": 6,
    "min_child_weight": 8,
    "learning_rate": 0.051612,
    "subsample": 0.988352,
    "colsample_bytree": 0.952685,
    "gamma": 0.089951,
    "reg_alpha": 0.285222,
    "reg_lambda": 2.843754
}

model_multi_tuned = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=1500,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
    **best_params
)

model_multi_tuned.fit(
    X_multi_train,
    y_multi_train,
    eval_set=[(X_multi_val, y_multi_val)],
    verbose=False
)

print("Best iteration:", model_multi_tuned.best_iteration)

Best iteration: 41


In [29]:
y_multi_val_tuned_pred = model_multi_tuned.predict(X_multi_val)

val_tuned_results = []

for horizon in sorted(df_multi.loc[X_multi_val.index, "horizon_minutes"].unique()):
    mask = df_multi.loc[X_multi_val.index, "horizon_minutes"] == horizon

    y_true = y_multi_val.loc[mask]
    y_pred = y_multi_val_tuned_pred[mask]

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    val_tuned_results.append({
        "horizon_minutes": horizon,
        "MAE": mae,
        "RMSE": rmse
    })

val_tuned_results = pd.DataFrame(val_tuned_results)

print(val_tuned_results.to_string(index=False))

 horizon_minutes       MAE      RMSE
              15 11.132288 13.785892
              30 13.935861 17.319985
              60 19.866855 24.986429
             120 27.246880 35.410028
             180 32.858466 42.915081
             360 45.304195 58.181055


In [30]:
y_multi_test_tuned_pred = model_multi_tuned.predict(X_multi_test)

test_tuned_results = []

for horizon in sorted(df_multi.loc[X_multi_test.index, "horizon_minutes"].unique()):
    mask = df_multi.loc[X_multi_test.index, "horizon_minutes"] == horizon

    y_true = y_multi_test.loc[mask]
    y_pred = y_multi_test_tuned_pred[mask]

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    test_tuned_results.append({
        "horizon_minutes": horizon,
        "MAE": mae,
        "RMSE": rmse
    })

test_tuned_results = pd.DataFrame(test_tuned_results)

print(test_tuned_results.to_string(index=False))

 horizon_minutes       MAE      RMSE
              15 13.688233 16.800819
              30 17.494377 21.223749
              60 24.583017 29.956216
             120 33.645519 42.730875
             180 40.385093 52.469211
             360 46.414565 63.833360


In [31]:
importance_df = pd.DataFrame({
    "feature": X_multi_train.columns,
    "importance": model_multi_tuned.feature_importances_
})

importance_df = importance_df.sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

print(importance_df.to_string(index=False))

                  feature  importance
          active_vehicles    0.503910
                  lag_15m    0.232244
                     hour    0.023040
    rolling_mean_exits_1h    0.017830
          horizon_minutes    0.017213
              day_of_week    0.015757
                   lag_6h    0.015370
               is_weekend    0.015148
     occupancy_change_15m    0.013728
                 exits_1h    0.012118
                  lag_24h    0.010436
 rolling_mean_arrivals_1h    0.010361
              arrivals_1h    0.009615
                exits_30m    0.009332
rolling_mean_occupancy_1h    0.009244
 rolling_mean_net_flow_1h    0.008677
                   lag_4h    0.008569
      occupancy_change_1h    0.008327
                   lag_2h    0.008179
      occupancy_change_2h    0.008177
                   lag_3h    0.007778
             net_flow_30m    0.006125
              net_flow_1h    0.005328
                   lag_1h    0.004829
                exits_15m    0.004540
            

In [32]:
pd.set_option("display.max_rows", None)

print(importance_df.to_string(index=False))

                  feature  importance
          active_vehicles    0.503910
                  lag_15m    0.232244
                     hour    0.023040
    rolling_mean_exits_1h    0.017830
          horizon_minutes    0.017213
              day_of_week    0.015757
                   lag_6h    0.015370
               is_weekend    0.015148
     occupancy_change_15m    0.013728
                 exits_1h    0.012118
                  lag_24h    0.010436
 rolling_mean_arrivals_1h    0.010361
              arrivals_1h    0.009615
                exits_30m    0.009332
rolling_mean_occupancy_1h    0.009244
 rolling_mean_net_flow_1h    0.008677
                   lag_4h    0.008569
      occupancy_change_1h    0.008327
                   lag_2h    0.008179
      occupancy_change_2h    0.008177
                   lag_3h    0.007778
             net_flow_30m    0.006125
              net_flow_1h    0.005328
                   lag_1h    0.004829
                exits_15m    0.004540
            

In [33]:
for i, row in importance_df.iterrows():
    print(f"{i + 1:2d}. {row['feature']:<35} {row['importance']:.6f}")

 1. active_vehicles                     0.503910
 2. lag_15m                             0.232244
 3. hour                                0.023040
 4. rolling_mean_exits_1h               0.017830
 5. horizon_minutes                     0.017213
 6. day_of_week                         0.015757
 7. lag_6h                              0.015370
 8. is_weekend                          0.015148
 9. occupancy_change_15m                0.013728
10. exits_1h                            0.012118
11. lag_24h                             0.010436
12. rolling_mean_arrivals_1h            0.010361
13. arrivals_1h                         0.009615
14. exits_30m                           0.009332
15. rolling_mean_occupancy_1h           0.009244
16. rolling_mean_net_flow_1h            0.008677
17. lag_4h                              0.008569
18. occupancy_change_1h                 0.008327
19. lag_2h                              0.008179
20. occupancy_change_2h                 0.008177
21. lag_3h          

In [34]:
from sklearn.inspection import permutation_importance

perm_result = permutation_importance(
    model_multi_tuned,
    X_multi_val,
    y_multi_val,
    scoring="neg_mean_absolute_error",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

permutation_df = pd.DataFrame({
    "feature": X_multi_val.columns,
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std
})

permutation_df = permutation_df.sort_values(
    "importance_mean",
    ascending=False
).reset_index(drop=True)

for i, row in permutation_df.iterrows():
    print(
        f"{i + 1:2d}. "
        f"{row['feature']:<35} "
        f"{row['importance_mean']:.6f} "
        f"+/- {row['importance_std']:.6f}"
    )

 1. active_vehicles                     27.255478 +/- 0.559540
 2. hour                                3.975538 +/- 0.232049
 3. horizon_minutes                     3.095666 +/- 0.116528
 4. day_of_week                         0.356011 +/- 0.085075
 5. occupancy_change_15m                0.205262 +/- 0.046549
 6. lag_6h                              0.154084 +/- 0.030276
 7. lag_15m                             0.139390 +/- 0.031465
 8. lag_2h                              0.043715 +/- 0.006817
 9. exits_1h                            0.042142 +/- 0.045249
10. arrivals_1h                         0.041839 +/- 0.010297
11. occupancy_change_2h                 0.032362 +/- 0.008680
12. lag_1h                              0.027353 +/- 0.008780
13. arrivals_30m                        0.022058 +/- 0.010323
14. exits_30m                           0.008746 +/- 0.015864
15. net_flow_15m                        0.006406 +/- 0.001394
16. net_flow_1h                         0.006297 +/- 0.002766
17. rol

In [35]:
selected_features = [
    "active_vehicles",
    "hour",
    "horizon_minutes",
    "day_of_week",
    "occupancy_change_15m",
    "lag_6h",
    "lag_15m",
    "lag_2h",
    "exits_1h",
    "arrivals_1h",
    "occupancy_change_2h",
    "lag_1h",
    "arrivals_30m",
    "exits_30m",
    "net_flow_15m",
    "net_flow_1h",
]

print("Selected features:", len(selected_features))
print(selected_features)

Selected features: 16
['active_vehicles', 'hour', 'horizon_minutes', 'day_of_week', 'occupancy_change_15m', 'lag_6h', 'lag_15m', 'lag_2h', 'exits_1h', 'arrivals_1h', 'occupancy_change_2h', 'lag_1h', 'arrivals_30m', 'exits_30m', 'net_flow_15m', 'net_flow_1h']


In [36]:
X_reduced_train = X_multi_train[selected_features]
X_reduced_val = X_multi_val[selected_features]
X_reduced_test = X_multi_test[selected_features]

print("Reduced train:", X_reduced_train.shape)
print("Reduced validation:", X_reduced_val.shape)
print("Reduced test:", X_reduced_test.shape)

Reduced train: (17631, 16)
Reduced validation: (3774, 16)
Reduced test: (3801, 16)


In [37]:
model_multi_reduced = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=1500,
    learning_rate=0.051612,
    max_depth=6,
    min_child_weight=8,
    subsample=0.988352,
    colsample_bytree=0.952685,
    gamma=0.089951,
    reg_alpha=0.285222,
    reg_lambda=2.843754,
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
    early_stopping_rounds=50
)

model_multi_reduced.fit(
    X_reduced_train,
    y_multi_train,
    eval_set=[(X_reduced_val, y_multi_val)],
    verbose=False
)

print("Best iteration:", model_multi_reduced.best_iteration)

Best iteration: 41


In [38]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

# Validation predictions
val_pred_reduced = model_multi_reduced.predict(X_reduced_val)

# Test predictions
test_pred_reduced = model_multi_reduced.predict(X_reduced_test)

# Keep horizon information from the original split
val_eval = X_multi_val[["horizon_minutes"]].copy()
val_eval["actual"] = y_multi_val.values
val_eval["prediction"] = val_pred_reduced

test_eval = X_multi_test[["horizon_minutes"]].copy()
test_eval["actual"] = y_multi_test.values
test_eval["prediction"] = test_pred_reduced


def calculate_metrics(df):
    results = []

    for horizon, group in df.groupby("horizon_minutes"):
        mae = mean_absolute_error(
            group["actual"],
            group["prediction"]
        )

        rmse = np.sqrt(
            mean_squared_error(
                group["actual"],
                group["prediction"]
            )
        )

        results.append({
            "horizon_minutes": horizon,
            "MAE": mae,
            "RMSE": rmse
        })

    return pd.DataFrame(results).sort_values("horizon_minutes")


reduced_val_results = calculate_metrics(val_eval)
reduced_test_results = calculate_metrics(test_eval)

print("Validation:")
display(reduced_val_results)

print("\nTest:")
display(reduced_test_results)

Validation:


,horizon_minutes,MAE,RMSE
0,15,11.140154,13.813981
1,30,13.865495,17.339976
2,60,19.722890,24.938518
3,120,27.387269,35.507876
4,180,32.820100,42.789482
5,360,45.023885,57.721883



Test:


,horizon_minutes,MAE,RMSE
0,15,14.049230,17.163644
1,30,17.772847,21.518635
2,60,24.589758,30.087741
3,120,34.036899,43.380658
4,180,40.659563,53.223991
5,360,46.627507,64.132936


In [39]:
%pip install lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 39.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [40]:
import lightgbm as lgb

print("LightGBM version:", lgb.__version__)

LightGBM version: 4.7.0


In [41]:
from lightgbm import LGBMRegressor

model_lgbm = LGBMRegressor(
    objective="regression",
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

print(model_lgbm)

LGBMRegressor(colsample_bytree=0.8, learning_rate=0.05, max_depth=6,
              n_estimators=1000, n_jobs=-1, objective='regression',
              random_state=42, reg_lambda=1.0, subsample=0.8)


In [42]:
model_lgbm = LGBMRegressor(
    objective="regression",
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

model_lgbm.fit(
    X_multi_train,
    y_multi_train,
    eval_set=[(X_multi_val, y_multi_val)],
    callbacks=[
        lgb.early_stopping(50, verbose=False)
    ]
)

print("Best iteration:", model_lgbm.best_iteration_)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.024222 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4458
[LightGBM] [Info] Number of data points in the train set: 17631, number of used features: 29
[LightGBM] [Info] Start training from score 167.894674
Best iteration: 37


/mnt/mydrive/ssk_group/myenv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


In [43]:
# Validation predictions
val_pred_lgbm = model_lgbm.predict(
    X_multi_val,
    num_iteration=model_lgbm.best_iteration_
)

# Test predictions
test_pred_lgbm = model_lgbm.predict(
    X_multi_test,
    num_iteration=model_lgbm.best_iteration_
)

# Validation evaluation
lgbm_val_eval = X_multi_val[["horizon_minutes"]].copy()
lgbm_val_eval["actual"] = y_multi_val.values
lgbm_val_eval["prediction"] = val_pred_lgbm

# Test evaluation
lgbm_test_eval = X_multi_test[["horizon_minutes"]].copy()
lgbm_test_eval["actual"] = y_multi_test.values
lgbm_test_eval["prediction"] = test_pred_lgbm


def calculate_lgbm_metrics(df):
    results = []

    for horizon, group in df.groupby("horizon_minutes"):
        mae = mean_absolute_error(
            group["actual"],
            group["prediction"]
        )

        rmse = np.sqrt(
            mean_squared_error(
                group["actual"],
                group["prediction"]
            )
        )

        results.append({
            "horizon_minutes": horizon,
            "MAE": mae,
            "RMSE": rmse
        })

    return pd.DataFrame(results).sort_values("horizon_minutes")


lgbm_val_results = calculate_lgbm_metrics(lgbm_val_eval)
lgbm_test_results = calculate_lgbm_metrics(lgbm_test_eval)

print("LightGBM Validation:")
display(lgbm_val_results)

print("\nLightGBM Test:")
display(lgbm_test_results)

LightGBM Validation:


,horizon_minutes,MAE,RMSE
0,15,11.994535,14.806292
1,30,14.553474,18.107218
2,60,20.139463,25.193006
3,120,27.679055,35.757696
4,180,33.199455,42.724483
5,360,44.871115,57.177146



LightGBM Test:


,horizon_minutes,MAE,RMSE
0,15,15.456293,19.130838
1,30,18.646548,22.941563
2,60,24.987353,30.708694
3,120,33.535234,42.696816
4,180,39.770467,51.847318
5,360,45.615720,63.157303


In [44]:
param_distributions_lgbm = {
    "num_leaves": randint(15, 64),
    "max_depth": randint(3, 10),
    "learning_rate": uniform(0.02, 0.08),
    "min_child_samples": randint(10, 80),
    "subsample": uniform(0.7, 0.3),
    "colsample_bytree": uniform(0.7, 0.3),
    "reg_alpha": uniform(0.0, 0.5),
    "reg_lambda": uniform(0.5, 4.5)
}

In [45]:
from scipy.stats import randint, uniform

param_distributions_lgbm = {
    "num_leaves": randint(15, 64),
    "max_depth": randint(3, 10),
    "learning_rate": uniform(0.02, 0.08),
    "min_child_samples": randint(10, 80),
    "subsample": uniform(0.7, 0.3),
    "colsample_bytree": uniform(0.7, 0.3),
    "reg_alpha": uniform(0.0, 0.5),
    "reg_lambda": uniform(0.5, 4.5)
}

print("LightGBM search space:")
for parameter, distribution in param_distributions_lgbm.items():
    print(f"{parameter}: {distribution}")

LightGBM search space:
num_leaves: <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x75a4503117f0>
max_depth: <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x75a49bdd5590>
learning_rate: <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x75a4503024e0>
min_child_samples: <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x75a49bdd57b0>
subsample: <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x75a49be52270>
colsample_bytree: <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x75a49be53a10>
reg_alpha: <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x75a49be682c0>
reg_lambda: <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x75a49bdb9e50>


In [46]:
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error
from scipy.stats import randint, uniform

rng = np.random.RandomState(42)

search_results_lgbm = []

for i in range(30):

    params = {
        "num_leaves": randint(15, 64).rvs(random_state=rng),
        "max_depth": randint(3, 10).rvs(random_state=rng),
        "learning_rate": uniform(0.02, 0.08).rvs(random_state=rng),
        "min_child_samples": randint(10, 80).rvs(random_state=rng),
        "subsample": uniform(0.7, 0.3).rvs(random_state=rng),
        "colsample_bytree": uniform(0.7, 0.3).rvs(random_state=rng),
        "reg_alpha": uniform(0.0, 0.5).rvs(random_state=rng),
        "reg_lambda": uniform(0.5, 4.5).rvs(random_state=rng)
    }

    model = LGBMRegressor(
        objective="regression",
        n_estimators=1500,
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
        subsample_freq=1,
        **params
    )

    model.fit(
        X_multi_train,
        y_multi_train,
        eval_set=[(X_multi_val, y_multi_val)],
        callbacks=[
            lgb.early_stopping(50, verbose=False)
        ]
    )

    val_pred = model.predict(
        X_multi_val,
        num_iteration=model.best_iteration_
    )

    val_mae = mean_absolute_error(
        y_multi_val,
        val_pred
    )

    search_results_lgbm.append({
        "iteration": i + 1,
        "validation_mae": val_mae,
        "best_iteration": model.best_iteration_,
        **params
    })

    print(
        f"Iteration {i + 1:02d}/30 | "
        f"Validation MAE: {val_mae:.4f} | "
        f"Best iteration: {model.best_iteration_}"
    )

search_results_lgbm = pd.DataFrame(search_results_lgbm)

search_results_lgbm = search_results_lgbm.sort_values(
    "validation_mae"
).reset_index(drop=True)

print("\nTop 10 configurations:")
display(search_results_lgbm.head(10))

/mnt/mydrive/ssk_group/myenv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Iteration 01/30 | Validation MAE: 25.1405 | Best iteration: 21


/mnt/mydrive/ssk_group/myenv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Iteration 02/30 | Validation MAE: 25.9264 | Best iteration: 84


/mnt/mydrive/ssk_group/myenv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


...

/mnt/mydrive/ssk_group/myenv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Iteration 29/30 | Validation MAE: 25.3151 | Best iteration: 69


/mnt/mydrive/ssk_group/myenv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Iteration 30/30 | Validation MAE: 26.2271 | Best iteration: 155

Top 10 configurations:


,iteration,validation_mae,best_iteration,num_leaves,max_depth,learning_rate,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda
0,18,24.975000,20,39,9,0.092546,10,0.768381,0.828132,0.409007,4.373288
1,27,25.097809,23,49,9,0.072799,69,0.778249,0.998876,0.482710,3.012321
2,1,25.140467,21,53,6,0.096057,70,0.879055,0.833750,0.049987,2.566620
3,19,25.272815,29,53,8,0.060860,18,0.845449,0.907731,0.134706,1.598565
4,17,25.274373,23,20,6,0.084294,53,0.824953,0.964984,0.162173,1.049396
5,6,25.283538,58,16,6,0.039282,69,0.736611,0.848553,0.017194,4.591942
6,16,25.310395,68,50,7,0.026158,68,0.908435,0.964140,0.312177,1.830352
7,29,25.315104,69,38,6,0.026731,45,0.969566,0.881929,0.004599,0.956622
8,4,25.345809,68,29,8,0.027249,71,0.759902,0.854270,0.296207,0.709027
9,21,25.368304,43,63,7,0.042787,68,0.882869,0.850804,0.025739,1.753909


In [47]:
best_lgbm_params = {
    "num_leaves": 39,
    "max_depth": 9,
    "learning_rate": 0.092546,
    "min_child_samples": 10,
    "subsample": 0.768381,
    "colsample_bytree": 0.828132,
    "reg_alpha": 0.409007,
    "reg_lambda": 4.373288
}

model_lgbm_tuned = LGBMRegressor(
    objective="regression",
    n_estimators=1500,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
    subsample_freq=1,
    **best_lgbm_params
)

model_lgbm_tuned.fit(
    X_multi_train,
    y_multi_train,
    eval_set=[(X_multi_val, y_multi_val)],
    callbacks=[
        lgb.early_stopping(50, verbose=False)
    ]
)

print("Best iteration:", model_lgbm_tuned.best_iteration_)

Best iteration: 20


/mnt/mydrive/ssk_group/myenv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


In [48]:
# Predictions
val_pred_lgbm_tuned = model_lgbm_tuned.predict(
    X_multi_val,
    num_iteration=model_lgbm_tuned.best_iteration_
)

test_pred_lgbm_tuned = model_lgbm_tuned.predict(
    X_multi_test,
    num_iteration=model_lgbm_tuned.best_iteration_
)

# Build evaluation tables
lgbm_tuned_val_eval = X_multi_val[["horizon_minutes"]].copy()
lgbm_tuned_val_eval["actual"] = y_multi_val.values
lgbm_tuned_val_eval["prediction"] = val_pred_lgbm_tuned

lgbm_tuned_test_eval = X_multi_test[["horizon_minutes"]].copy()
lgbm_tuned_test_eval["actual"] = y_multi_test.values
lgbm_tuned_test_eval["prediction"] = test_pred_lgbm_tuned


def calculate_metrics(df):
    results = []

    for horizon, group in df.groupby("horizon_minutes"):
        mae = mean_absolute_error(
            group["actual"],
            group["prediction"]
        )

        rmse = np.sqrt(
            mean_squared_error(
                group["actual"],
                group["prediction"]
            )
        )

        results.append({
            "horizon_minutes": horizon,
            "MAE": mae,
            "RMSE": rmse
        })

    return pd.DataFrame(results).sort_values("horizon_minutes")


lgbm_tuned_val_results = calculate_metrics(
    lgbm_tuned_val_eval
)

lgbm_tuned_test_results = calculate_metrics(
    lgbm_tuned_test_eval
)

print("Tuned LightGBM — Validation:")
display(lgbm_tuned_val_results)

print("\nTuned LightGBM — Test:")
display(lgbm_tuned_test_results)

Tuned LightGBM — Validation:


,horizon_minutes,MAE,RMSE
0,15,11.514141,14.285395
1,30,14.368795,17.925921
2,60,20.419402,25.690487
3,120,27.558711,35.701748
4,180,32.826160,42.691833
5,360,43.162819,55.418191



Tuned LightGBM — Test:


,horizon_minutes,MAE,RMSE
0,15,14.718939,18.343106
1,30,18.086508,22.373610
2,60,24.761974,30.338520
3,120,33.721105,42.545957
4,180,39.833989,51.891458
5,360,46.408648,63.421024


In [49]:
persistence_pred = X_multi_test["active_vehicles"]

In [50]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

persistence_results = []

for horizon in sorted(X_multi_test["horizon_minutes"].unique()):
    mask = X_multi_test["horizon_minutes"] == horizon

    y_actual = y_multi_test[mask]
    y_pred = persistence_pred[mask]

    mae = mean_absolute_error(y_actual, y_pred)
    rmse = np.sqrt(mean_squared_error(y_actual, y_pred))

    persistence_results.append({
        "horizon_minutes": horizon,
        "mae": mae,
        "rmse": rmse
    })

persistence_results = pd.DataFrame(persistence_results)

persistence_results

,horizon_minutes,mae,rmse
0,15,8.984399,11.136719
1,30,15.009375,18.702105
2,60,23.846395,30.010291
3,120,34.982650,45.325131
4,180,43.433333,56.550287
5,360,59.100324,78.585142


In [51]:
train_baseline = df_multi.loc[
    df_multi["timestamp"] < pd.Timestamp("2026-09-05 04:45:00")
].copy()

train_baseline["target_time"] = (
    train_baseline["timestamp"]
    + pd.to_timedelta(train_baseline["horizon_minutes"], unit="m")
)

hourly_occupancy_mean = (
    train_baseline
    .groupby(train_baseline["target_time"].dt.hour)["target"]
    .mean()
)

hourly_occupancy_mean

target_time
0     152.624828
1     133.885246
2     125.945652
3     125.783784
4     120.471622
5     125.047814
6     147.741450
7     154.716644
8     154.298343
9     153.238950
10    161.600277
11    171.658333
12    177.526897
13    186.221311
14    210.676630
15    214.012162
16    208.367568
17    198.229730
18    196.322581
19    191.338710
20    176.096774
21    176.782258
22    190.862903
23    173.620968
Name: target, dtype: float64

In [52]:
target_time_test = (
    X_multi_test["timestamp"]
    + pd.to_timedelta(X_multi_test["horizon_minutes"], unit="m")
)

historical_mean_pred = target_time_test.dt.hour.map(
    hourly_occupancy_mean
)

historical_mean_results = []

for horizon in sorted(X_multi_test["horizon_minutes"].unique()):
    mask = X_multi_test["horizon_minutes"] == horizon

    y_actual = y_multi_test[mask]
    y_pred = historical_mean_pred[mask]

    mae = mean_absolute_error(y_actual, y_pred)
    rmse = np.sqrt(mean_squared_error(y_actual, y_pred))

    historical_mean_results.append({
        "horizon_minutes": horizon,
        "mae": mae,
        "rmse": rmse
    })

historical_mean_results = pd.DataFrame(historical_mean_results)

historical_mean_results

KeyError: 'timestamp'

In [53]:
df_multi.columns

Index(['timestamp', 'active_vehicles', 'lag_15m', 'lag_30m', 'lag_1h',
       'lag_2h', 'lag_3h', 'lag_4h', 'lag_6h', 'lag_24h', 'arrivals_15m',
       'arrivals_30m', 'arrivals_1h', 'exits_15m', 'exits_30m', 'exits_1h',
       'net_flow_15m', 'net_flow_30m', 'net_flow_1h', 'occupancy_change_15m',
       'occupancy_change_1h', 'occupancy_change_2h',
       'rolling_mean_occupancy_1h', 'rolling_mean_arrivals_1h',
       'rolling_mean_exits_1h', 'rolling_mean_net_flow_1h', 'hour',
       'day_of_week', 'is_weekend', 'horizon_minutes', 'target'],
      dtype='str')

In [54]:
test_mask = df_multi["timestamp"] >= pd.Timestamp("2026-09-11 18:00:00")

df_multi_test = df_multi.loc[test_mask].copy()

print("Test rows:", len(df_multi_test))
print("Test start:", df_multi_test["timestamp"].min())
print("Test end:", df_multi_test["timestamp"].max())

Test rows: 3801
Test start: 2026-09-11 18:00:00
Test end: 2026-09-18 10:00:00


In [55]:
target_time_test = (
    df_multi_test["timestamp"]
    + pd.to_timedelta(df_multi_test["horizon_minutes"], unit="m")
)

historical_mean_pred = target_time_test.dt.hour.map(
    hourly_occupancy_mean
)

historical_mean_results = []

for horizon in sorted(df_multi_test["horizon_minutes"].unique()):
    mask = df_multi_test["horizon_minutes"] == horizon

    y_actual = df_multi_test.loc[mask, "target"]
    y_pred = historical_mean_pred[mask]

    mae = mean_absolute_error(y_actual, y_pred)
    rmse = np.sqrt(mean_squared_error(y_actual, y_pred))

    historical_mean_results.append({
        "horizon_minutes": horizon,
        "mae": mae,
        "rmse": rmse
    })

historical_mean_results = pd.DataFrame(historical_mean_results)

historical_mean_results

,horizon_minutes,mae,rmse
0,15,45.227298,57.117840
1,30,45.295345,57.162407
2,60,45.414382,57.250219
3,120,45.653589,57.426531
4,180,45.885030,57.602539
5,360,46.401015,58.052527


In [56]:
occupancy_lookup = (
    df_multi[["timestamp", "active_vehicles"]]
    .drop_duplicates("timestamp")
    .set_index("timestamp")["active_vehicles"]
)

print("Occupancy timestamps:", len(occupancy_lookup))
print("Start:", occupancy_lookup.index.min())
print("End:", occupancy_lookup.index.max())

Occupancy timestamps: 4216
Start: 2026-08-02 00:00:00
End: 2026-09-18 10:00:00


In [57]:
target_time_test = (
    df_multi_test["timestamp"]
    + pd.to_timedelta(df_multi_test["horizon_minutes"], unit="m")
)

previous_day_time = target_time_test - pd.Timedelta(hours=24)

previous_day_pred = previous_day_time.map(occupancy_lookup)

print("Missing previous-day predictions:", previous_day_pred.isna().sum())

Missing previous-day predictions: 0


In [58]:
previous_day_results = []

for horizon in sorted(df_multi_test["horizon_minutes"].unique()):
    mask = df_multi_test["horizon_minutes"] == horizon

    y_actual = df_multi_test.loc[mask, "target"]
    y_pred = previous_day_pred[mask]

    mae = mean_absolute_error(y_actual, y_pred)
    rmse = np.sqrt(mean_squared_error(y_actual, y_pred))

    previous_day_results.append({
        "horizon_minutes": horizon,
        "mae": mae,
        "rmse": rmse
    })

previous_day_results = pd.DataFrame(previous_day_results)

previous_day_results

,horizon_minutes,mae,rmse
0,15,60.536661,74.807981
1,30,60.515625,74.809237
2,60,60.451411,74.787997
3,120,60.498423,74.899596
4,180,60.455556,74.943481
5,360,61.001618,75.465792


In [59]:
import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [60]:
df_multi = pd.read_csv("supervised_multihorizon.csv")

df_multi["timestamp"] = pd.to_datetime(df_multi["timestamp"])

print("Shape:", df_multi.shape)
print("Columns:", df_multi.columns.tolist())

Shape: (25206, 31)
Columns: ['timestamp', 'active_vehicles', 'lag_15m', 'lag_30m', 'lag_1h', 'lag_2h', 'lag_3h', 'lag_4h', 'lag_6h', 'lag_24h', 'arrivals_15m', 'arrivals_30m', 'arrivals_1h', 'exits_15m', 'exits_30m', 'exits_1h', 'net_flow_15m', 'net_flow_30m', 'net_flow_1h', 'occupancy_change_15m', 'occupancy_change_1h', 'occupancy_change_2h', 'rolling_mean_occupancy_1h', 'rolling_mean_arrivals_1h', 'rolling_mean_exits_1h', 'rolling_mean_net_flow_1h', 'hour', 'day_of_week', 'is_weekend', 'horizon_minutes', 'target']


In [61]:
train_end = pd.Timestamp("2026-09-05 04:45:00")
val_end = pd.Timestamp("2026-09-11 18:00:00")

train_mask = df_multi["timestamp"] < train_end
val_mask = (
    (df_multi["timestamp"] >= train_end)
    & (df_multi["timestamp"] < val_end)
)
test_mask = df_multi["timestamp"] >= val_end

train_df = df_multi.loc[train_mask].copy()
val_df = df_multi.loc[val_mask].copy()
test_df = df_multi.loc[test_mask].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (17631, 31)
Validation: (3774, 31)
Test: (3801, 31)


In [62]:
feature_columns = [
    col for col in df_multi.columns
    if col not in ["timestamp", "target"]
]

target_column = "target"

X_train = train_df[feature_columns]
y_train = train_df[target_column]

X_val = val_df[feature_columns]
y_val = val_df[target_column]

X_test = test_df[feature_columns]
y_test = test_df[target_column]

print("Number of features:", len(feature_columns))
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

Number of features: 29
X_train: (17631, 29)
X_val: (3774, 29)
X_test: (3801, 29)


In [64]:
model_p50 = XGBRegressor(
    objective="reg:quantileerror",
    quantile_alpha=0.50,

    n_estimators=1500,
    learning_rate=0.051612,
    max_depth=6,
    min_child_weight=8,

    subsample=0.988352,
    colsample_bytree=0.952685,

    gamma=0.089951,
    reg_alpha=0.285222,
    reg_lambda=2.843754,

    random_state=42,
    n_jobs=-1,
    tree_method="hist",

    early_stopping_rounds=50
)

In [65]:
model_p50.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print("Best iteration:", model_p50.best_iteration)

Best iteration: 177


In [66]:
model_p90 = XGBRegressor(
    objective="reg:quantileerror",
    quantile_alpha=0.90,

    n_estimators=1500,
    learning_rate=0.051612,
    max_depth=6,
    min_child_weight=8,

    subsample=0.988352,
    colsample_bytree=0.952685,

    gamma=0.089951,
    reg_alpha=0.285222,
    reg_lambda=2.843754,

    random_state=42,
    n_jobs=-1,
    tree_method="hist",

    early_stopping_rounds=50
)

In [70]:
model_p90.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print("Best iteration:", model_p90.best_iteration)

Best iteration: 229


In [71]:
p50_pred = model_p50.predict(X_test)
p90_pred = model_p90.predict(X_test)

print("P50 predictions:", p50_pred[:10])
print("P90 predictions:", p90_pred[:10])

P50 predictions: [211.05855 186.29572 186.32384 175.21153 186.26335 171.70045 204.25452
 195.27745 185.40434 180.29018]
P90 predictions: [229.40076 214.02785 211.14871 198.57808 207.31245 194.16098 219.7528
 218.7587  213.06596 197.54396]


In [73]:
invalid_order = (p90_pred < p50_pred).sum()

print("P90 < P50:", invalid_order)
print("Total predictions:", len(p50_pred))

P90 < P50: 5
Total predictions: 3801


In [74]:
p50_mae = mean_absolute_error(y_test, p50_pred)

print("P50 MAE:", p50_mae)

P50 MAE: 27.99862321242945


In [75]:
def pinball_loss(y_true, y_pred, quantile):
    error = y_true - y_pred

    return np.mean(
        np.where(
            error >= 0,
            quantile * error,
            (quantile - 1) * error
        )
    )

In [76]:
p50_pinball = pinball_loss(
    y_test,
    p50_pred,
    0.50
)

p90_pinball = pinball_loss(
    y_test,
    p90_pred,
    0.90
)

print("P50 pinball loss:", p50_pinball)
print("P90 pinball loss:", p90_pinball)

P50 pinball loss: 13.999311606214725
P90 pinball loss: 6.889625611640189


In [77]:
p90_coverage = np.mean(y_test <= p90_pred)

print("P90 coverage:", p90_coverage)
print("P90 coverage (%):", p90_coverage * 100)

P90 coverage: 0.8945014469876348
P90 coverage (%): 89.45014469876348


In [78]:
quantile_results = []

for horizon in sorted(test_df["horizon_minutes"].unique()):

    mask = test_df["horizon_minutes"].values == horizon

    y_actual = y_test.values[mask]
    p50 = p50_pred[mask]
    p90 = p90_pred[mask]

    quantile_results.append({
        "horizon_minutes": horizon,

        "p50_mae": mean_absolute_error(
            y_actual,
            p50
        ),

        "p50_pinball": pinball_loss(
            y_actual,
            p50,
            0.50
        ),

        "p90_pinball": pinball_loss(
            y_actual,
            p90,
            0.90
        ),

        "p90_coverage": np.mean(
            y_actual <= p90
        )
    })

quantile_results = pd.DataFrame(quantile_results)

quantile_results

,horizon_minutes,p50_mae,p50_pinball,p90_pinball,p90_coverage
0,15,9.696186,4.848093,2.356753,0.950078
1,30,15.100191,7.550096,3.368461,0.909375
2,60,23.810731,11.905365,5.489431,0.885580
3,120,34.183144,17.091572,8.039717,0.870662
4,180,39.710080,19.855040,9.847556,0.857143
5,360,46.379740,23.189870,12.487986,0.893204


In [79]:
quantile_results = []

for horizon in sorted(test_df["horizon_minutes"].unique()):

    mask = test_df["horizon_minutes"].values == horizon

    y_actual = y_test.values[mask]
    p50 = p50_pred[mask]
    p90 = p90_pred[mask]

    quantile_results.append({
        "horizon_minutes": horizon,

        "p50_mae": mean_absolute_error(
            y_actual,
            p50
        ),

        "p50_pinball": pinball_loss(
            y_actual,
            p50,
            0.50
        ),

        "p90_pinball": pinball_loss(
            y_actual,
            p90,
            0.90
        ),

        "p90_coverage": np.mean(
            y_actual <= p90
        )
    })

quantile_results = pd.DataFrame(quantile_results)

quantile_results

,horizon_minutes,p50_mae,p50_pinball,p90_pinball,p90_coverage
0,15,9.696186,4.848093,2.356753,0.950078
1,30,15.100191,7.550096,3.368461,0.909375
2,60,23.810731,11.905365,5.489431,0.885580
3,120,34.183144,17.091572,8.039717,0.870662
4,180,39.710080,19.855040,9.847556,0.857143
5,360,46.379740,23.189870,12.487986,0.893204
